# Stage 1: Non-Instruction Fine-Tuning

**Project:** IT Helpdesk AI Assistant (domain-specific fine-tuning with Unsloth)

**Goal:** Adapt a small open-source base model to IT Helpdesk domain language, terminology, and writing style using raw (non-instruction) text, *before* instruction tuning.

> **Run this notebook on a GPU runtime** (Google Colab free T4, or Kaggle GPU). Unsloth requires a CUDA GPU; it will not run on CPU-only machines.

Steps covered:
1. Load raw domain text (`data/non_instruction_data.txt`)
2. Clean and chunk the text
3. Load the base model with Unsloth
4. Apply LoRA/QLoRA
5. Train on raw text (causal language modeling)
6. Save the adapter/model
7. Test the model after non-instruction fine-tuning

## 0. Install dependencies (Colab / Kaggle GPU runtime only)

In [ ]:
# !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl peft accelerate bitsandbytes

## 1. Load raw domain text

In [ ]:
from pathlib import Path

# When running on Colab, upload/clone the repo first, or download the raw file directly.
DATA_PATH = Path("../data/non_instruction_data.txt")
raw_text = DATA_PATH.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} raw domain paragraphs")
print(paragraphs[0])

## 2. Clean and chunk the text

We collapse whitespace, drop very short paragraphs, and pack paragraphs into fixed-length chunks suitable for causal LM training.

In [ ]:
import re

def clean(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned = [clean(p) for p in paragraphs if len(p.split()) >= 15]
print(f"{len(cleaned)} paragraphs kept after cleaning")

MAX_CHARS = 512
chunks = []
buffer = ""
for p in cleaned:
    if len(buffer) + len(p) + 1 > MAX_CHARS:
        if buffer:
            chunks.append(buffer)
        buffer = p
    else:
        buffer = f"{buffer} {p}".strip()
if buffer:
    chunks.append(buffer)

print(f"Built {len(chunks)} training chunks")
print(chunks[0])

## 3. Load base model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/tinyllama-bnb-4bit"  # TinyLlama-1.1B, 4-bit quantized
MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

## 3b. Baseline: evaluate the untouched base model (before any fine-tuning)

Run the 10 fixed evaluation questions through the raw base model (no LoRA, no training yet) and save the answers to `reports/base_model_answers.json`. `src/fill_reports.py` uses this file to auto-populate `reports/base_model_evaluation.md`.

In [ ]:
import json
import sys

sys.path.append("../src")
from eval_questions import EVAL_QUESTIONS

FastLanguageModel.for_inference(model)

BASE_PROMPT_TEMPLATE = "{question}\n"

base_answers = []
for q in EVAL_QUESTIONS:
    inputs = tokenizer(BASE_PROMPT_TEMPLATE.format(question=q), return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded[len(BASE_PROMPT_TEMPLATE.format(question=q)):].strip()
    base_answers.append({"question": q, "answer": answer})
    print("Q:", q)
    print("A:", answer)
    print("-" * 80)

with open("../reports/base_model_answers.json", "w", encoding="utf-8") as f:
    json.dump(base_answers, f, indent=2)
print("Saved baseline answers to reports/base_model_answers.json")

## 4. Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 5. Train on raw text (non-instruction / continued pre-training)

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = Dataset.from_dict({"text": chunks})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_non_instruction",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 6. Save the adapter

In [ ]:
SAVE_DIR = "models/non_instruction_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved non-instruction LoRA adapter to {SAVE_DIR}")

## 7. Test the model after non-instruction fine-tuning

We expect the model to now produce IT-helpdesk-style phrasing more naturally, even though it has not yet been instruction-tuned to directly answer questions (that happens in Stage 2).

In [ ]:
FastLanguageModel.for_inference(model)

prompt = "Multiple Device Connection Problems on Corporate VPN."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))